# Lab 2 · Read a Postcode, Pythonically

*Same answers first, prettier second, and measured.*

**Your mission:** Rewrite the intern's digit reader as short, readable Python, show with golden checks that it gives exactly the same answers, measure what changed, and protect it with a test.

**Reaching Putting It All Together (the end of the minimum path) is a complete lab. Everything after it is extra.**

**In this notebook**

1. Setup: run the first code cell
2. How this lab works: the markers ✏️ WRITE CODE and ▶️ JUST RUN, and the three parts CORE, OPTIONAL and ADVANCED
3. Previously on Garabato: a two-minute recap
4. What is refactoring? Same answer first, prettier second
5. Just enough theory: tuples, dicts, comprehensions, zip, sorted and max with a key
6. Fast lane: the whole CORE on one table
7. Load the data, meet the legacy reader, and the worked example W
8. Warm-up: predict the output
9. CORE: C1 → C2 → C3 → C4 → Putting It All Together → C5
10. Check yourself: four exam-style questions, on paper (5 minutes)
11. After the CORE, with the whole class: AI wrote this: find the planted bugs
12. OPTIONAL (extra practice): C6, ENG1, SCI1, BLD1
13. ADVANCED (for students who move faster): ENG2, SCI2, BLD2, the Challenge: The Postcode Machine
14. AI-use declaration, Before you go, What's next

Data: MNIST (LeCun, Cortes, Burges), CC BY-SA 3.0, adapted. Garabato Parcels is fictional.

**Today you review Theory S3–S4 (data structures and Python idioms: tuples, dicts, comprehensions, `zip`, `enumerate`, `sorted` and `max` with a key) and your first tests with `assert`, by rewriting the Lab 1 reader the Pythonic way.**

In [ ]:
# ▶️ JUST RUN · Run me first (Shift+Enter). You don't need to read this cell.
LAB = 2
KIT_URL = "https://github.com/xavitel/programming-lab/releases/download/lab2/lab2_kit.zip"   # filled in at build time if the kit is hosted; used only on Colab
import os, sys, platform
from pathlib import Path

def _find_kit(start):
    """Return the folder containing labkit/: here, up to 3 levels above, or 1 level below."""
    start = Path(start).resolve()
    for folder in [start] + list(start.parents)[:3]:
        if (folder / "labkit" / "__init__.py").is_file():
            return folder
    for child in sorted(p for p in start.iterdir() if p.is_dir()):
        if (child / "labkit" / "__init__.py").is_file():
            return child
    return None

print("Python", platform.python_version(), "|", sys.executable)
if sys.version_info < (3, 10):
    raise SystemExit("STOP: this kernel runs Python " + platform.python_version()
                     + ". The lab needs Python 3.10 or newer.\nFix: click the kernel name (top right) > "
                       "Select Another Kernel > Python Environments > pick 3.10+ (e.g. your course .venv). "
                       "START_HERE.md, Route A step 3, shows exactly where to click.")
if ".zip" in str(Path.cwd()).lower():
    raise SystemExit("STOP: the notebook was opened from inside the zip file.\nFix: close it, right-click "
                     "the zip > Extract All, then in VS Code use File > Open Folder on the extracted folder.")
IN_COLAB = "google.colab" in sys.modules
kit = _find_kit(Path.cwd())
if kit is None and IN_COLAB:
    import zipfile, urllib.request
    zips = sorted(Path("/content").glob("lab*_kit*.zip"))
    if not zips and KIT_URL:
        print("Colab: downloading the kit ...")
        urllib.request.urlretrieve(KIT_URL, "/content/kit.zip")
        zips = [Path("/content/kit.zip")]
    for z in zips:
        zipfile.ZipFile(z).extractall("/content")
    kit = _find_kit("/content")
if kit is None:
    raise SystemExit("STOP: I can't find the labkit folder.\nFix: open the WHOLE extracted kit folder in "
                     "VS Code (File > Open Folder), not only the notebook. On Colab, upload the kit zip "
                     "with the Files panel and run this cell again.")
os.chdir(kit)
if str(kit) not in sys.path:
    sys.path.insert(0, str(kit))
try:
    get_ipython().run_line_magic("load_ext", "autoreload")   # picks up edits in .py files
    get_ipython().run_line_magic("autoreload", "2")
except Exception:
    pass
import labkit
labkit.start(LAB)
from labkit import check, check_all, rescue, load_digits, show_digit, compare, stopwatch

## How this lab works

**Two markers tell you what to do with every cell:**

| Marker | What it means |
|---|---|
| **✏️ WRITE CODE** | You write or change code here. Stubs start with `# ===== YOUR CODE HERE =====`: replace `pass` with your code. |
| **▶️ JUST RUN** | Provided code: run it (Shift+Enter) and read what it prints. You don't need to change it. |

Helper functions in `labkit` are already implemented: use them, don't modify them.

**Three parts:**

| Part | Who and when | Feedback |
|---|---|---|
| **CORE** | everyone, in order: C1 → C2 → C3 → C4 → Putting It All Together (the minimum path: reaching it is a complete lab), then C5 | `check()`, 4 hints, `rescue()` |
| **OPTIONAL** | extra practice when your CORE is done, any task, any order: C6, ENG1, SCI1, BLD1 | `check()` and hints |
| **ADVANCED** | for students who move faster; open-ended, you can continue at home: ENG2, SCI2, BLD2, the Challenge: The Postcode Machine | a sanity check, a score, or your own measurements |

**Three tools.**
- `check("C1", squared_distance)` answers PASS, NOT YET or NOT STARTED. It shows the tests it ran (TEST 1, TEST 2, …),
  with the inputs, the expected result and your result; on NOT YET, the first failing test and a likely cause.
- **Hints** sit under each task. Open them one at a time.
- `squared_distance = rescue("C1")` hands you a working version, so the rest of the notebook keeps working.

**Stuck for 7 minutes after Hint 3? Use `rescue()`**, carry on, and come back to the task later.

**The AI dial** (a recommendation, not a ban):
- CORE: AI as **tutor**. Ask *why*, not for the answer; Hint 4 gives you a prompt. Try on your own for 7–10 minutes first.
- OPTIONAL and ADVANCED tasks: AI as **reviewer**. Keep only what you can explain line by line.
- The Challenge (ADVANCED): AI as **pair programmer**. Verify with checks and tests, and declare it at the end.
- Always: you own every line you run, you verify it, and you **never paste personal data into an AI tool**.

**Help card:** show **HELP PLEASE** when you'd like help, **WORKING** otherwise.
Work alone or with a neighbour, your choice. Nothing here is graded or collected.

## Previously on Garabato

Two minutes of recap. You don't need any code from earlier labs.

**The story so far:** Lab 1 *I can code* → **Lab 2 (today)** *I can write good Python*

**Last time, in Lab 1:** you taught Python to read one handwritten digit with functions you wrote yourself: `squared_distance`, nearest neighbour and `accuracy`, built from loops.

**The ideas you keep using:**

- An image is a list of 784 numbers (0 = paper, 255 = ink).
- Distance = the sum of squared differences.
- Nearest neighbour = copy the label of the closest known image.
- k nearest neighbours vote = ask the k closest images and take the most common label (Lab 1 ADVANCED, so new for most of you; today it is in the CORE).
- Centroid = the average image of each digit (Lab 1 OPTIONAL; here only in the Challenge).
- Accuracy = the fraction right on images the model never saw.
- Missed a lab? Everything you need is in `labkit.reference`.

## What is refactoring?

**The story.** Garabato's summer intern left a digit reader, `legacy_reader.py` (in your kit folder). It works:
it reads a digit by letting its **3 nearest training images vote** (Lab 1's nearest neighbour, with a vote).
But it is long, full of counters and copied code, and nobody dares to touch it. Ops also want whole postcodes,
not single digits, so the code has to grow. Before it grows, it must become code people can read and change.

**Refactoring** means changing how code is written without changing what it does: the same inputs give the same
outputs, before and after. Three rules for today:

1. **Same answer first, prettier second.** A refactor that changes one answer is a bug, however elegant it looks.
2. **Compare with a golden check.** Run the old code and the new code on the same inputs and compare every
   answer: the old answers are the "golden" ones. `compare(old, new)` does it for you.
3. **Measure, don't guess.** Is the new version shorter? Faster? Count the lines and time both versions.

**Why touch code that works?** Because some ways of writing it make it risky to change. Each one has a Python
idiom that removes the risk (Theory S3–S4):

| What makes code risky to change | Where in `legacy_reader.py` | What you'll write instead |
|---|---|---|
| A counter you manage yourself | `i = 0` … `while i < 784:` … `i = i + 1` | `for` over the values, `zip`, a comprehension |
| A magic number in the code | `784`, `128`, `3` | a parameter: `threshold=128`, `k=3` |
| Copy-paste | the distance loop, copied into `k_nearest` | call `squared_distance` |
| A sort written by hand | a bubble sort of 1,000 pairs | `sorted(pairs, key=...)` |
| Counting by hand | two parallel lists, `labels` and `counts` | a dict, and `max(counts, key=counts.get)` |

**The plan of this lab, in six sentences**

| | Where |
|---|---|
| I can spot what makes old code risky. | *Meet the legacy reader* |
| I can rewrite a loop as one readable line and show it gives the same answers. | W and C1 |
| I can sort with a key and count with a dict. | C2 and C3 |
| I can compose small functions into a reader. | C4 |
| I can show that the whole new reader matches the old one, and measure both. | Putting It All Together |
| I can write a test that catches bugs. | C5 |

**Short on time?** The theory primers and the warm-up are optional: each task card names the theory it needs.

## Just enough theory

Optional: skip this section if you've done Theory S3–S4. Open a primer only if the idea feels new; each one is a short example you can run in your head, or paste into a new cell and run.

<details>
<summary><strong>Tuples and unpacking (Theory S3)</strong></summary>

A tuple groups a few values and never changes. Unpacking gives each value a name.

```python
pair = (25, 7)                    # (distance, label)
distance, label = pair            # distance is 25, label is 7
for distance, label in [(25, 7), (9, 1)]:
    print(label)                  # 7, then 1
```

</details>

<details>
<summary><strong>Counting with a dict (Theory S3)</strong></summary>

One key per thing you count. A dict keeps its keys in the order they were first added.

```python
counts = {}
for label in [7, 1, 7]:
    counts[label] = counts.get(label, 0) + 1
print(counts)                     # {7: 2, 1: 1}
```

</details>

<details>
<summary><strong>List comprehensions and generator expressions (Theory S4)</strong></summary>

A whole loop that builds a list, in one line. Inside `sum()`, `min()` or `max()` you can drop the brackets.

```python
[x * x for x in [1, 2, 3]]                 # [1, 4, 9]
[x for x in [1, 2, 3, 4] if x % 2 == 0]    # [2, 4]: with a condition
sum(x * x for x in [1, 2, 3])              # 14: a generator expression, added up
```

</details>

<details>
<summary><strong>zip and enumerate (Theory S4)</strong></summary>

`zip` walks several lists side by side; `enumerate` numbers the items as it goes.

```python
for x, y in zip([1, 2, 3], [10, 20, 30]):
    print(x + y)                           # 11, 22, 33
```

`enumerate`, compact and step by step. Both loops print the same two lines, `1 fig`, then `2 kiwi`:

```python
fruits = ["fig", "kiwi"]

# Compact: enumerate hands out the counter and the element together
for rank, fruit in enumerate(fruits, start=1):
    print(rank, fruit)

# Step by step: the same output, with a counter you manage yourself
rank = 1                                   # start the counter (that is start=1)
for fruit in fruits:
    print(rank, fruit)
    rank = rank + 1                        # increase it by hand: forget this line and every rank is 1
```

`fruits` is the list. On each pass of the loop, `fruit` receives one element of the list (`"fig"`, then `"kiwi"`)
and `rank` receives the counter (1, then 2). In the compact version `enumerate` produces the pairs `(1, "fig")` and
`(2, "kiwi")`, and `for rank, fruit in ...` unpacks each pair: no counter to start, increase or forget.

</details>

<details>
<summary><strong>sorted, min and max with a key, and lambda (Theory S4)</strong></summary>

The key is a function: Python sorts (or picks) by what it returns, not by the elements themselves.

```python
words = ["kiwi", "fig", "apple"]
sorted(words, key=len)                     # ['fig', 'kiwi', 'apple']: shortest first
max(words, key=len)                        # 'apple'
```

**The key as a named function.** `key=last_letter` passes the function itself (no brackets). `sorted` then calls
it once for each element of the list, and that element becomes the value of the parameter `word`. What the function
returns is what `sorted` compares:

```python
def last_letter(word):
    print("sorted() called last_letter with", word)
    return word[-1]                        # the value sorted() compares for this word

print(sorted(words, key=last_letter))
# sorted() called last_letter with kiwi    -> returns 'i'
# sorted() called last_letter with fig     -> returns 'g'
# sorted() called last_letter with apple   -> returns 'e'
# ['apple', 'fig', 'kiwi']                 because 'e' < 'g' < 'i'
```

**The same key as a lambda.** `lambda word: word[-1]` is the same function without a name: the parameter before
the colon, the returned value after it. `sorted` calls it in exactly the same way, once per word:

```python
print(sorted(words, key=lambda word: word[-1]))    # ['apple', 'fig', 'kiwi']
```

In C2, `key=lambda pair: pair[0]` receives one `(distance, label)` pair at a time and returns its distance, so the
pairs are sorted by distance only.

</details>

## Fast lane

Some people like to read a spec and build; others like guided steps. Both routes reach the same place.

| Task | Signature | Spec |
|---|---|---|
| C1 (minimum path) | `squared_distance(a: list[int], b: list[int]) -> int` | the sum of (x - y) ** 2 over the pixel pairs that zip(a, b) gives, in one line |
| C2 (minimum path) | `k_nearest(image: list[int], train_images: list[list[int]], train_labels: list[int], k: int = 3) -> list[tuple[int, int]]` | (distance, label) for the k nearest training images, closest first; equal distances keep the training order (sort by the distance only) |
| C3 (minimum path) | `majority_vote(neighbours: list[tuple[int, int]]) -> int` | the most frequent label; a tie goes to the closest tied neighbour (the neighbours come closest first) |
| C4 (minimum path) | `predict_knn(image: list[int], train_images: list[list[int]], train_labels: list[int], k: int = 3) -> int` | the majority_vote of the k_nearest neighbours |
| C5 (not needed later) | `test_majority_vote(vote_fn) -> None` | asserts that call vote_fn on at least 3 inputs, one of them a tie: it passes on a correct vote and fails on buggy ones |

(minimum path) marks the tasks you need for Putting It All Together: C1 → C2 → C3 → C4. Each function has a stub cell further down, starting with `# ===== YOUR CODE HERE =====`: write your code there, run it, then run the cell below at any time to see where you are. Each line tells you the next step.

In [ ]:
# ▶️ JUST RUN · where am I?
check_all();   # the ; only stops Jupyter from printing the same result again as a dict

## Load the data and meet the legacy reader

This lab works with 1,000 examples of handwritten digits and another 100 images to test the program: the same
images as in Lab 1. Each image is a list of 784 numbers (0 = paper, 255 = ink), and its label is the digit it
shows. The cell also imports the intern's code as `legacy` (so `legacy.predict(...)` runs the old reader) and
the provided tools of this lab: they are already implemented, you only run them.

In [ ]:
# ▶️ JUST RUN · load the 1,000 training images and the 100 test images, the legacy reader and this lab's tools
train_images, train_labels = load_digits("data/mnist_train_10k.csv.gz", limit=1000)
test_images, test_labels = load_digits("data/mnist_test_2k.csv.gz", limit=100)
import legacy_reader as legacy                # the intern's code (read-only): legacy.predict, legacy.k_nearest, ...
from labkit.lab2_tools import (ready, show_checks, show_source, time_versions, show_line_counts, show_neighbours,
                               show_vote, run_test)
print(f"{len(train_images):,} training images and {len(test_images)} test images, "
      f"{len(train_images[0])} pixels each.")

### Meet the legacy reader (about 3 minutes)

Run the cell below: it prints `k_nearest` from `legacy_reader.py`, with line numbers. (You can also open the
file in VS Code; please don't edit it: the golden checks compare your code with it.) Read the code and find:

1. a **counter** that the code starts, tests and increases itself;
2. a **magic number**: a value written straight into the code;
3. **copy-paste**: code that already exists somewhere else in the file;
4. the **bubble sort**: how many comparisons does it make to sort 1,000 pairs?

In [ ]:
# ▶️ JUST RUN · the legacy k_nearest, with its line numbers in legacy_reader.py
show_source(legacy.k_nearest)

<details>
<summary><strong>Answers</strong></summary>

1. `i = 0`, `while i < 784:` and `i = i + 1` (and the same again with `a`, `b` and `j`): forget the last line
   and the loop never ends.
2. `784` (pixels; a 14 × 14 image has 196) and `3` (the neighbours: you can't ask for 5).
3. The distance loop is a copy of `squared_distance`: a bug fixed in one copy stays in the other.
4. The outer loop runs 999 times and the inner one a little less each time: 999 + 998 + … + 1 = 499,500
   comparisons for 1,000 pairs, for every digit it reads.

</details>

### W · Worked example: Refactor ink_count · ▶️ JUST RUN

The intern's ink_count needs a counter, a while loop and the magic number 784. Rewrite it as one readable line, then show that it gives the same answers.

**You'll practise:** a generator expression inside sum() (Theory S4)

**Signature**

```python
def ink_count(image: list[int], threshold: int = 128) -> int
```

**Toy example**

```python
ink_count([0, 128, 255, 127])    # 128 and 255 are ink -> 2
```

A worked example: the code is already in the next cell. Compare it with the legacy loop below, run it, then run
the **golden check**: `compare(legacy.ink_count, ink_count)` runs both versions on the same 20 test images, compares
every answer and times both. Then run the check.

```python
def ink_count(image):            # legacy_reader.py
    count = 0
    i = 0
    while i < 784:
        if image[i] >= 128:
            count = count + 1
        i = i + 1
    return count
```

In [ ]:
# ▶️ JUST RUN · the worked example: the legacy loop, rewritten as one line
def ink_count(image: list[int], threshold: int = 128) -> int:
    """Return how many pixels of `image` are ink: value >= threshold.

    >>> ink_count([0, 128, 255, 127])
    2
    """
    return sum(1 for pixel in image if pixel >= threshold)     # a 1 for every ink pixel, added up

In [ ]:
# ▶️ JUST RUN · the golden check: the old and the new version on the same 20 test images, compared and timed
golden = compare(legacy.ink_count, ink_count)

In [ ]:
# ▶️ JUST RUN · check your work
check("W", ink_count);

**Try it (1 minute).** Change `>=` to `>` in the cell with `def ink_count`, run that cell, then the golden
check again: it tells you the first test image where the two versions disagree. The check below it says
why. Change it back and run both again: **same answer first, prettier second.** And look at the two times:
which version was faster?

### Warm-up: predict the output

For while others finish their setup. Read each cell, **write down what you think it prints**, then run it.
The answers are under the third cell. Each one is a small preview of a CORE task.

In [ ]:
# ▶️ JUST RUN · warm-up 1 (predict first, then run)
pairs = [(9, 7), (4, 1), (4, 0)]
print(sorted(pairs))
print(sorted(pairs, key=lambda pair: pair[0]))

In [ ]:
# ▶️ JUST RUN · warm-up 2 (predict first, then run)
a, b = [1, 2, 3], [4, 6, 3]
print(list(zip(a, b)))
print(sum((x - y) ** 2 for x, y in zip(a, b)))

In [ ]:
# ▶️ JUST RUN · warm-up 3 (predict first, then run)
counts = {}
for label in [3, 8, 8, 3]:
    counts[label] = counts.get(label, 0) + 1
print(counts, max(counts), max(counts, key=counts.get))

<details>
<summary><strong>Answers</strong></summary>

1. `[(4, 0), (4, 1), (9, 7)]`, then `[(4, 1), (4, 0), (9, 7)]`. Without a key, tuples are compared item by
   item, so the two 4s are ordered by their second item. With `key=lambda pair: pair[0]` only the first item
   counts, and equal ones keep their order: (4, 1) came first. That is C2's tie rule.
2. `[(1, 4), (2, 6), (3, 3)]`, then `25`: 9 + 16 + 0. That is C1.
3. `{3: 2, 8: 2} 8 3`. `max(counts)` compares the keys, so it says 8, the largest label. With
   `key=counts.get` it compares the counts, 2 and 2, and returns the first key with the top count: 3. That is C3.

</details>

## CORE: the tasks everyone does

**C1 → C2 → C3 → C4 → Putting It All Together → C5.** The first five are the **minimum path**: reaching Putting It All Together is a complete lab. C5 completes the CORE.

For each task: read the card, write your code in the cell that starts with `# ===== YOUR CODE HERE =====` (replace `pass`), run that cell, then run the check under it. The check shows each test: what it checks, its inputs, the expected result and yours. The tests on real images are golden checks: your answer must equal the legacy reader's. The checks test what your function does, not how it is written, so a plain loop that gives the right answers passes too. (`check(..., style=True)` adds optional style notes, for example on `range(len(...))` and on `while` loops with a counter.)

### C1 · How different are two images? In one line · ✏️ WRITE CODE

Lab 1 measured the difference between two images with a running total. Write the same squared distance as one line: a sum over the pixels, paired by zip().

**You'll practise:** zip() and a generator expression inside sum() (Theory S4) · **Typical time:** 4–6 min

**Minimum path: you need this for Putting It All Together.**

**Signature**

```python
def squared_distance(a: list[int], b: list[int]) -> int
```

**Toy example**

```python
squared_distance([0, 0], [3, 4])        # (0 - 3) ** 2 + (0 - 4) ** 2 = 9 + 16 = 25
squared_distance([1, 2, 3], [1, 2, 3])  # identical images: 0
```

<details>
<summary><strong>The legacy code it replaces</strong> (from the tour)</summary>

```python
total = 0
i = 0
while i < 784:               # a counter to start, test and increase, and a magic number
    d = a[i] - b[i]
    total = total + d * d
    i = i + 1
return total
```

</details>

**Why one line?** Less to get wrong (no counter, no index, no 784), and the same arithmetic, so the same
answers.

**Stuck for 7 minutes after Hint 3?** Run `squared_distance = rescue("C1")` in a new cell, carry on, and come back to C1 later.

In [ ]:
# ===== YOUR CODE HERE =====
# ✏️ WRITE CODE: replace `pass` with your code, then run this cell (Shift+Enter).
def squared_distance(a: list[int], b: list[int]) -> int:
    """Return the squared distance between two images: the sum of (x - y) ** 2 over matching pixels.

    >>> squared_distance([0, 0], [3, 4])
    25
    """
    # TODO: in one line, return the sum(...) of (x - y) ** 2 for every pair x, y in zip(a, b)
    pass

In [ ]:
# ▶️ JUST RUN · check your work
check("C1", squared_distance);

In [ ]:
# ▶️ JUST RUN · measure, don't guess: one distance between two real images, written three ways (under 1 s)
from labkit.reference import lab1             # Lab 1's version: a for loop with a running total
time_versions({"legacy (while loop)": legacy.squared_distance,
               "Lab 1 (for loop)": lab1.squared_distance,
               "yours (sum and zip)": squared_distance},
              test_images[0], train_images[0])

**Was the one-liner the fastest?** Compare with your neighbour: another laptop or Python version may disagree.
Readable code is the goal of a refactor; speed is measured, not guessed (ENG1 goes further).

<details>
<summary><strong>Hint 1 · Nudge</strong></summary>

zip(a, b) gives you the pixels in pairs: x from a and y from b, position by position (Theory S4). For each pair you need (x - y) ** 2, and sum(...) adds them all up.

</details>

<details>
<summary><strong>Hint 2 · Plan</strong></summary>

1. `for x, y in zip(a, b)` walks both images side by side: pixel 0 of a with pixel 0 of b, and so on.
2. The squared difference of one pair is `(x - y) ** 2`.
3. Write the expression first and the `for` after it, all inside `sum( ... )`, and return the result.

Without zip (the Lab 1 loop passes too): `total = 0`, then `for i in range(len(a)):` add `(a[i] - b[i]) ** 2`, then `return total`.

</details>

<details>
<summary><strong>Hint 3 · Skeleton</strong></summary>

    return sum((___ - ___) ** 2 for ___, ___ in zip(___, ___))

</details>

<details>
<summary><strong>Hint 4 · Ask an AI tutor (not for the answer)</strong></summary>

Copy this into your AI assistant: *"I'm learning Python. Act as a tutor and do NOT write code for me. Here is my function and the check output: [paste]. Ask me one question at a time that helps me find the problem myself."*

</details>

> **Why let several neighbours vote?** One odd training image can fool a single nearest neighbour (Lab 1's
> reader). Ask the k nearest and take the most common label, and the odd one gets outvoted. C2 finds the k
> nearest, C3 counts their votes. After C3, the vote view shows test image #13: what does its nearest image say,
> and what does the vote say? k = 3 is the intern's choice (SCI2 asks how to choose it).

### C2 · The k closest pictures, sorted with a key · ✏️ WRITE CODE

The intern sorted the (distance, label) pairs with a hand-written bubble sort. Build the pairs with zip(), sort them with sorted() and a key, and keep the first k.

**You'll practise:** pairs as tuples, zip(), and sorted() with key= and a lambda (Theory S3–S4) · **Typical time:** 10–15 min

**Minimum path: you need this for Putting It All Together.**

**Signature**

```python
def k_nearest(image: list[int], train_images: list[list[int]], train_labels: list[int],
              k: int = 3) -> list[tuple[int, int]]
```

**Toy example**

```python
# 1-pixel images: the new image [5], then four training images and their labels
k_nearest([5], [[9], [4], [6], [0]], [7, 8, 3, 5], k=2)    # distances 16, 1, 1, 25
# -> [(1, 8), (1, 3)]: [4] and [6] tie, so they keep the training order (sorted(pairs) would swap them)
```

**Why `sorted(..., key=...)`?**

| | Legacy bubble sort | `sorted(pairs, key=lambda pair: pair[0])` |
|---|---|---|
| Code | two nested while loops and a swap | one call |
| Comparisons for 1,000 pairs | 499,500 | about 10,000 |
| Equal distances | keep the training order | keep it too (the sort is *stable*); without the key, `sorted(pairs)` compares the labels next |

**Stuck for 7 minutes after Hint 3?** Run `k_nearest = rescue("C2")` in a new cell, carry on, and come back to C2 later.

In [ ]:
# ===== YOUR CODE HERE =====
# ✏️ WRITE CODE: replace `pass` with your code, then run this cell (Shift+Enter).
def k_nearest(image: list[int], train_images: list[list[int]], train_labels: list[int],
              k: int = 3) -> list[tuple[int, int]]:
    """Return the k nearest training images as (distance, label) pairs, the closest first.

    Equal distances keep the order of the training data (sort by the distance only).

    >>> k_nearest([5], [[9], [4], [6], [0]], [7, 8, 3, 5], k=2)
    [(1, 8), (1, 3)]
    """
    # TODO 1: pairs = one (squared_distance(image, train), label) tuple per training image (zip helps)
    # TODO 2: sort the pairs by their distance only: sorted(pairs, key=...)
    # TODO 3: return the first k
    pass

In [ ]:
# ▶️ JUST RUN · check your work
check("C2", k_nearest);

**Your neighbours** (once C2 passes): a test image and the training images your `k_nearest` returns, with their
labels and distances.

In [ ]:
# ▶️ JUST RUN · a test image and its 3 nearest training images, found by YOUR k_nearest
test_index = 0                        # optional: try another test image, e.g. 13 or 14
if ready("k_nearest"):
    show_neighbours(test_images[test_index], test_labels[test_index], train_images, train_labels,
                    neighbours_fn=k_nearest, k=3)

<details>
<summary><strong>Hint 1 · Nudge</strong></summary>

Two lines (Theory S3–S4): a list comprehension that builds one (distance, label) tuple per training image, then sorted() with a key that looks at the distance only. A slice, [:k], keeps the first k.

</details>

<details>
<summary><strong>Hint 2 · Plan</strong></summary>

1. Build `pairs`: one tuple `(distance, label)` per training image. `zip(train_images, train_labels)` gives you each training image together with its label; the distance is `squared_distance(image, train)`.
2. Sort with `sorted(pairs, key=lambda pair: pair[0])`: the key looks at the distance only, so equal distances keep their order.
3. Return the first `k` pairs of the sorted list.

Without a comprehension (it passes too): `pairs = []`, then `for i in range(len(train_images)):` append `(squared_distance(image, train_images[i]), train_labels[i])`.

</details>

<details>
<summary><strong>Hint 3 · Skeleton</strong></summary>

    pairs = [(squared_distance(___, ___), ___) for train, label in zip(___, ___)]
    return sorted(pairs, key=lambda pair: ___)[:___]

</details>

<details>
<summary><strong>Hint 4 · Ask an AI tutor (not for the answer)</strong></summary>

Copy this into your AI assistant: *"I'm learning Python. Act as a tutor and do NOT write code for me. Here is my function and the check output: [paste]. Ask me one question at a time that helps me find the problem myself."*

</details>

### C3 · Let the neighbours vote · ✏️ WRITE CODE

The intern counted votes with two parallel lists. Count them with a dict and pick the winner with max() and a key; a tie goes to the closest tied neighbour.

**You'll practise:** counting with a dict, and max() with a key (Theory S3–S4) · **Typical time:** 8–12 min

**Minimum path: you need this for Putting It All Together.**

**Signature**

```python
def majority_vote(neighbours: list[tuple[int, int]]) -> int
```

**Toy example**

```python
majority_vote([(3, 7), (5, 1), (9, 7)])           # two 7s beat one 1 -> 7
majority_vote([(2, 4), (5, 1)])                   # a 1-1 tie: the closer neighbour wins -> 4
majority_vote([(1, 3), (2, 8), (4, 8), (6, 3)])   # a 2-2 tie: the 3 is closer (distance 1) -> 3
```

**Why `max(counts, key=counts.get)`?** The 2-2 tie above, counted closest first, gives `counts = {3: 2, 8: 2}`:

| Expression | max() compares | Result |
|---|---|---|
| `max(counts)` | the labels, 3 and 8 | `8`: the largest label, not a vote |
| `max(counts, key=counts.get)` | the counts, 2 and 2 | `3`: the first key with the top count, so the closest |

**Stuck for 7 minutes after Hint 3?** Run `majority_vote = rescue("C3")` in a new cell, carry on, and come back to C3 later.

In [ ]:
# ===== YOUR CODE HERE =====
# ✏️ WRITE CODE: replace `pass` with your code, then run this cell (Shift+Enter).
def majority_vote(neighbours: list[tuple[int, int]]) -> int:
    """Return the most frequent label among the neighbours (closest first); a tie goes to the closest.

    >>> majority_vote([(3, 7), (5, 1), (9, 7)])
    7
    """
    # TODO 1: counts = {}; for each (distance, label) in neighbours, add 1 to the count of that label
    # TODO 2: return the label with the biggest count: max(counts, key=...)
    pass

In [ ]:
# ▶️ JUST RUN · check your work
check("C3", majority_vote);

**The vote, live.** The 5 nearest training images of test image #13 vote (five, so that the vote shows more
clearly), and YOUR `majority_vote` names the winner. Which label wins? Run it, then try `test_index = 1`.

In [ ]:
# ▶️ JUST RUN · the 5 nearest training images of a test image vote; YOUR majority_vote announces the winner
test_index = 13                       # then try 1, 10 or 14
if ready("majority_vote"):
    show_vote(majority_vote, test_images[test_index], train_images, train_labels, k=5,
              truth=test_labels[test_index])

<details>
<summary><strong>Hint 1 · Nudge</strong></summary>

Counting with a dict is the Lab 1 pattern (Theory S3): counts[label] = counts.get(label, 0) + 1. The winner is the key with the biggest value: max() with key=counts.get (Theory S4).

</details>

<details>
<summary><strong>Hint 2 · Plan</strong></summary>

1. `counts = {}`; then `for distance, label in neighbours:` add 1 to `counts[label]` (`.get(label, 0) + 1` handles a new label).
2. Return `max(counts, key=counts.get)`: the key makes max() compare each label's count, not the label itself.
3. Keep the plain dict and the neighbour order: that is what sends a tie to the closest neighbour.

Without max() and a key (it passes too): find `top = max(counts.values())`, then walk the neighbours again, closest first, and return the first label whose count is `top`.

</details>

<details>
<summary><strong>Hint 3 · Skeleton</strong></summary>

    counts = {}
    for distance, label in neighbours:
        counts[___] = counts.get(___, 0) + ___
    return max(___, key=___)

</details>

<details>
<summary><strong>Hint 4 · Ask an AI tutor (not for the answer)</strong></summary>

Copy this into your AI assistant: *"I'm learning Python. Act as a tutor and do NOT write code for me. Here is my function and the check output: [paste]. Ask me one question at a time that helps me find the problem myself."*

</details>

### C4 · Read a digit: neighbours, then a vote · ✏️ WRITE CODE

Glue your own functions together, as the intern's predict does: the k nearest neighbours, then their vote.

**You'll practise:** composing your own functions (Theory S1) · **Typical time:** 2–4 min

**Minimum path: you need this for Putting It All Together.**

**Signature**

```python
def predict_knn(image: list[int], train_images: list[list[int]], train_labels: list[int],
                k: int = 3) -> int
```

**Toy example**

```python
predict_knn([5], [[0], [4], [6], [7], [30]], [2, 1, 9, 9, 1], k=3)
# the three nearest are (1, 1), (1, 9), (4, 9): labels 1, 9, 9 -> 9
predict_knn([5], [[0], [4], [6], [7], [30]], [2, 1, 9, 9, 1], k=1)   # only the nearest -> 1
```

**Why so short?** Each step already has its own function and its own check, so `predict_knn` only connects them.
The TODO lines give the three steps: get the neighbours, compute the winner, return it. Writing them as three
lines is fine; once it passes, you can compose them into one: the result of one function goes straight into the
next.

**Stuck for 7 minutes after Hint 3?** Run `predict_knn = rescue("C4")` in a new cell, carry on, and come back to C4 later.

In [ ]:
# ===== YOUR CODE HERE =====
# ✏️ WRITE CODE: replace `pass` with your code, then run this cell (Shift+Enter).
def predict_knn(image: list[int], train_images: list[list[int]], train_labels: list[int], k: int = 3) -> int:
    """Read one digit: the k nearest training images vote.

    >>> predict_knn([5], [[0], [4], [6], [7], [30]], [2, 1, 9, 9, 1], k=3)
    9
    """
    # TODO 1: get the neighbours: the k nearest training images, from k_nearest (pass k on)
    # TODO 2: compute the winner: the majority_vote of those neighbours
    # TODO 3: return the winner
    pass

In [ ]:
# ▶️ JUST RUN · check your work
check("C4", predict_knn);

<details>
<summary><strong>Hint 1 · Nudge</strong></summary>

Three steps, each one call of a function you already wrote: what k_nearest returns is exactly what majority_vote needs.

</details>

<details>
<summary><strong>Hint 2 · Plan</strong></summary>

1. Get the neighbours: `neighbours = k_nearest(image, train_images, train_labels, k)`. Pass `k` on, so that k=1 really asks one neighbour.
2. Compute the winner: `winner = majority_vote(neighbours)`.
3. Return it: `return winner`.

Once it passes, the three lines can become one: `majority_vote(...)` can take the call `k_nearest(...)` directly.

</details>

<details>
<summary><strong>Hint 3 · Skeleton</strong></summary>

    # 1. get the neighbours (pass k on)
    neighbours = k_nearest(___, ___, ___, ___)
    # 2. compute the winner
    winner = majority_vote(___)
    # 3. return it
    return ___

    # or, composed into one line:
    # return majority_vote(k_nearest(___, ___, ___, ___))

</details>

<details>
<summary><strong>Hint 4 · Ask an AI tutor (not for the answer)</strong></summary>

Copy this into your AI assistant: *"I'm learning Python. Act as a tutor and do NOT write code for me. Here is my function and the check output: [paste]. Ask me one question at a time that helps me find the problem myself."*

</details>

### Putting It All Together · ▶️ JUST RUN

This is where your four functions replace the intern's reader, and you check that they give the same answers:

1. `squared_distance` (C1) **measures** how different two images are, in one line;
2. `k_nearest` (C2) **finds the closest** training images, with `sorted()` and a key;
3. `majority_vote` (C3) **lets them vote**, with a dict;
4. `predict_knn` (C4) **puts the two steps together**.

The cell first lists the status of your C1-C4 checks. Then `compare(legacy.predict, predict_knn, n=10)`
(provided), the golden check of the whole reader, runs both readers on the same 10 test images, each
compared with 1,000 training images, and compares every answer; then it counts the lines of code of both readers.
Same answers? Faster or slower? How much shorter? Run it and find out. It takes a few seconds.

In [ ]:
# ▶️ JUST RUN · Putting It All Together: your checks, then the legacy reader and YOUR reader on the same 10 test images
if ready("squared_distance", "k_nearest", "majority_vote", "predict_knn"):
    show_checks(squared_distance, k_nearest, majority_vote, predict_knn)
    golden = compare(legacy.predict, predict_knn, n=10)
    show_line_counts({
        "legacy_reader.py": [legacy.squared_distance, legacy.k_nearest, legacy.majority_vote, legacy.predict],
        "yours": [squared_distance, k_nearest, majority_vote, predict_knn]})

**Think about it.** Which reader would you rather debug at 3 a.m.? Was yours faster or slower than the legacy
one, and is that true on your neighbour's laptop too? (The C1 timing cell holds a clue.) Agreeing on 10
images is evidence, not proof: the checks of C1-C4 also cover rare cases, such as ties, that real images seldom
hit. A test is only good if it fails on bad code: that is C5.

**Reaching Putting It All Together is a complete lab.** Next: C5 completes the CORE with a test that catches bugs. Then, with
the whole class, *AI wrote this*; then the OPTIONAL and ADVANCED tasks.

### Live demo: a test is only good if it fails on bad code · ▶️ JUST RUN, then ✏️ WRITE CODE: add one assert (optional) · whole class, about 4 minutes

An `assert` states one fact. If the fact is true, nothing happens; if it is false, Python stops with an
`AssertionError` and your message:

```python
assert 2 + 2 == 4, "arithmetic is broken"         # true: nothing happens
assert sorted([3, 1]) == [3, 1], "not sorted"     # false: AssertionError: not sorted
```

A **test function** is a function full of asserts. It takes the function to test as a parameter, so the
same test can check any version: yours, the reference, or a buggy one. The cell below runs one small test
on two versions: your `majority_vote`, and a buggy vote that trusts only the closest neighbour. Run it and
read the result: did the test catch the bug? If not, work out an input where the closest label loses the
vote, add one assert with it to `test_demo` in the same cell, and run it again. (Still working on C1-C4?
Keep going, and run this cell later.)

In [ ]:
# ▶️ JUST RUN first. Then add one assert to test_demo so that it catches closest_only, and run it again.
# One test, two versions: yours and a buggy one (labkit.mutants holds code that is wrong on purpose)
from labkit.mutants.lab2 import closest_only       # a buggy vote: it returns the closest neighbour's label

def test_demo(vote_fn):
    assert vote_fn([(1, 7), (2, 7), (3, 1)]) == 7, "two 7s beat one 1"

run_test(test_demo, majority_vote, "your majority_vote")
run_test(test_demo, closest_only, "the buggy vote closest_only", buggy=True)

<details>
<summary><strong>One assert that catches it (open after you've tried)</strong></summary>

The first assert passes on the buggy vote too, because the closest neighbour (a 7) is also the winner. An
input where the closest label loses the vote catches it:
`assert vote_fn([(1, 1), (2, 7), (3, 7)]) == 7, "two 7s beat the closest 1"`. A test that passes on buggy
code gives false confidence: it has to fail when the code is wrong.

</details>

### C5 · A test that catches bugs · ✏️ WRITE CODE

Before anyone touches majority_vote again, protect it: write asserts that pass on a correct vote and fail on buggy ones.

**You'll practise:** assert, and testing a function you pass in (new tools: see the demo above) · **Typical time:** 8–12 min

**Not needed later:** nothing after this task depends on it, so you can come back to it after class.

**Signature**

```python
def test_majority_vote(vote_fn) -> None
```

**Toy example**

```python
test_majority_vote(majority_vote)     # your C3: every assert passes, nothing is printed
test_majority_vote(closest_only)      # the buggy vote of the demo: an assert fails (AssertionError)
```

**The rules:** check at least 3 inputs (3 assert lines, or one assert in a loop over 3 cases); every assert calls
`vote_fn(...)`, never `majority_vote(...)` directly; one input is a tie. `check("C5", test_majority_vote)` runs
your test on a correct vote (it must pass) and on 4 buggy votes (it must catch at least 3; they live in
`labkit/mutants/lab2.py`: try without peeking), and names in words every buggy vote that slips through. **Why a
tie?** Some bugs only show on a tie. **No brackets** around a whole assert: `assert (a == b, "message")` is a
tuple, which is always true, so it can never fail.

**Stuck for 7 minutes after Hint 3?** Run `test_majority_vote = rescue("C5")` in a new cell, carry on, and come back to C5 later.

In [ ]:
# ===== YOUR CODE HERE =====
# ✏️ WRITE CODE: replace `pass` with your code, then run this cell (Shift+Enter).
def test_majority_vote(vote_fn) -> None:
    """Test a majority-vote function: it passes on a correct vote and fails on a buggy one."""
    # TODO: at least 3 inputs, each checked by an assert that calls vote_fn on neighbours you write, e.g.
    #       assert vote_fn([(1, 7), (2, 7), (3, 1)]) == 7, "two 7s beat one 1"
    #       Include a tie, and a majority that is neither the closest label nor the largest one.
    pass

In [ ]:
# ▶️ JUST RUN · check your work
check("C5", test_majority_vote);

In [ ]:
# ▶️ JUST RUN · your test on your own majority_vote (C3), and on the buggy vote of the demo
from labkit.mutants.lab2 import closest_only       # the buggy vote of the demo
if ready("test_majority_vote", "majority_vote"):
    run_test(test_majority_vote, majority_vote, "your majority_vote")
    run_test(test_majority_vote, closest_only, "the buggy vote closest_only", buggy=True)

<details>
<summary><strong>Hint 1 · Nudge</strong></summary>

A test is a list of facts you are sure of. Work out a few small votes by hand (2 to 4 neighbours), then write each one as assert vote_fn(neighbours) == answer, "message".

</details>

<details>
<summary><strong>Hint 2 · Plan</strong></summary>

1. A clear majority whose winner is neither the closest label nor the largest one: `[(1, 9), (2, 4), (3, 4)]` must give 4.
2. A 1-1 tie, where the closer neighbour must win: `[(1, 7), (2, 3)]` gives 7.
3. The same tie the other way round, or a single neighbour.

Write each as `assert vote_fn(...) == ..., "message"`, with no brackets around the whole assert.

</details>

<details>
<summary><strong>Hint 3 · Skeleton</strong></summary>

    assert vote_fn([(1, 9), (2, 4), (3, 4)]) == ___, "two 4s beat one 9"
    assert vote_fn([(1, 7), (2, 3)]) == ___, "a 1-1 tie goes to the closer neighbour"
    assert vote_fn(___) == ___, "___"

</details>

<details>
<summary><strong>Hint 4 · Ask an AI tutor (not for the answer)</strong></summary>

Copy this into your AI assistant: *"I'm learning Python. Act as a tutor and do NOT write code for me. Here is my function and the check output: [paste]. Ask me one question at a time that helps me find the problem myself."*

</details>

**CORE complete.** Well done: a shorter reader, checked against the old one, measured, and protected by a test. Next,
*Check yourself* (5 minutes, on paper), then, with the whole class, *AI wrote this*. Then any OPTIONAL task,
or ADVANCED if you move faster.

## Check yourself (5 minutes, exam-style)

Four exam-style questions on today's Python (Theory S3–S4 and `assert`). Answer on paper or in your head first, then open the answer: nothing to run, nothing is checked. Still working on the CORE? Keep going, and come back to these at home.

**1. What does this print?**

```python
names = ["Cris", "Ben", "Ana"]
scores = [7, 9]
print([f"{name}:{score}" for name, score in zip(names, scores)])
for rank, name in enumerate(sorted(names, key=len), start=1):
    print(rank, name)
```

<details>
<summary><strong>Answer</strong></summary>

`['Cris:7', 'Ben:9']`, then `1 Ben`, `2 Ana`, `3 Cris`. `zip` stops at the shorter list, so `Ana` gets no
score. `sorted(..., key=len)` compares the lengths only: `Ben` and `Ana` both have 3 letters, so they keep their
order (Ben came first; it is not alphabetical). `enumerate(..., start=1)` counts from 1. C2 uses the same rule
for equal distances.

</details>

**2. What is wrong here? It runs, but `majority_vote([(1, 7), (2, 7), (3, 1)])` returns 1.**

```python
def majority_vote(neighbours):
    for distance, label in neighbours:
        counts = {}
        counts[label] = counts.get(label, 0) + 1
    return max(counts, key=counts.get)
```

<details>
<summary><strong>Answer</strong></summary>

`counts = {}` is inside the loop, so every neighbour starts a new, empty dict: only the last neighbour's vote
survives, and the function returns the farthest label. Create the dict once, before the `for`.

</details>

**3. Which structure would you use, and why? (a) The neighbours of a test image: a distance and a label each, to be sorted by distance. (b) The number of votes for each label.**

<details>
<summary><strong>Answer</strong></summary>

(a) A list of tuples, `[(distance, label), ...]`: a tuple keeps the two values of one neighbour together, so
sorting moves them as one. With two parallel lists, sorting one list breaks the link to the other. (b) A dict,
`{label: count}`: `counts.get(label, 0) + 1` updates a label in one step. The intern's two parallel lists,
`labels` and `counts`, first had to search for the label's position.

</details>

**4. What is wrong with this test? (Two things.)**

```python
def test_vote(vote_fn):
    assert (vote_fn([(1, 7), (2, 7), (3, 1)]) == 1, "two 7s beat one 1")
```

<details>
<summary><strong>Answer</strong></summary>

The brackets turn the assert into a tuple, `(comparison, "message")`, and a non-empty tuple is always true, so
this assert can never fail (Python even warns: *assertion is always true*). That hides the second mistake: the
expected answer should be 7, not 1. Write `assert vote_fn([(1, 7), (2, 7), (3, 1)]) == 7, "two 7s beat one 1"`:
a test is only good if it can fail.

</details>

## After the CORE: one activity with the whole class

It is done by everyone at the same time, so you can join it wherever you are in the CORE.

### AI wrote this: find the planted bugs · ▶️ JUST RUN in class (steps 1-2) · ✏️ WRITE CODE at home (step 3)

Someone asked an AI assistant for a k-NN reader and pasted its answer into the code base. It looks
professional: a docstring, comments, `zip`, a dict. **It has three bugs.** (The code is illustrative: it
was written for this lab in the style of an AI assistant's answer. It is not real AI output.)

1. **Read it first** (the cell below prints it with line numbers). Which lines would you question? Write
   down your guesses before you run anything else.
2. **Run its check:** it shows the first test the code fails, and a question that points at the bug.
   Name the three suspect lines (their line numbers are enough).
3. **Then, if you have time, or at home: fix the copy** in the cell after it, one bug at a time, running
   that cell and its check after each fix, until the check passes.

Still working on C1-C4 when the class starts this? Keep going: finish C4 and Putting It All Together first, and do this
part at home.

Would you have shipped it if it had passed one happy-path test? (ENG2, in the ADVANCED part, turns each bug
into a test, so it can never come back.)

In [ ]:
# ▶️ JUST RUN · the AI's answer, with line numbers
from labkit.ai_samples.lab2_knn_ai import knn_predict_ai
show_source(knn_predict_ai)

In [ ]:
# ▶️ JUST RUN · check the AI's code as it is: it fails on purpose, so read the first failing test
check("AI", knn_predict_ai);

In [ ]:
# ===== YOUR CODE HERE =====
# ✏️ WRITE CODE: fix the three bugs in this copy of the AI's code, then run this cell (Shift+Enter).
def knn_predict_ai(image, train_images, train_labels, k=3):
    """Predict the label of `image` using k-nearest neighbours (the AI's code: fix it)."""
    # Pair each training label with its distance to the query image
    pairs = []
    for label, train_image in zip(train_labels, train_images):
        distance = sum((p - q) ** 2 for p, q in zip(image, train_image))
        pairs.append((label, distance))

    # Sort the candidates so that the nearest neighbours come first
    pairs.sort()

    # Skip the first element: it is the query image itself (distance 0)
    nearest = pairs[1:k + 1]

    # Count the votes for each label
    votes = {}
    for label, distance in nearest:
        votes[label] = votes.get(label, 0) + 1

    # Return the label with the most votes
    return max(votes)

In [ ]:
# ▶️ JUST RUN · check your work
check("AI", knn_predict_ai);   # your fixed copy

<details>
<summary><strong>What we found (open after you've tried)</strong></summary>

1. `pairs.append((label, distance))` with `pairs.sort()`: tuples sort by their first item, so the
   "nearest" neighbours were the smallest **labels**. Put the distance first and sort by it alone,
   `key=lambda pair: pair[0]`: a bare `pairs.sort()` compares the labels when two distances are equal.
2. `nearest = pairs[1:k + 1]` skips the real nearest neighbour. The comment sounds reasonable, but the test
   image is not in the training data: there is nothing to skip.
3. `max(votes)` returns the largest label. The most votes is `max(votes, key=votes.get)`.

If you put the distance first, also unpack `for distance, label in nearest` in the vote loop, or it counts
the distances.

Each bug is plausible, and each comment sounds confident. Only a check (or a test) with small inputs whose
answers you know shows them.

</details>

## OPTIONAL: extra practice

Extra practice for when your CORE is done. Pick any task, in any order: each one is independent, has a `check()` and hints, and nothing later depends on it.

| Task | What you'll learn | Difficulty | Typical time |
|---|---|---|---|
| **C6** Scale the pixels | a list comprehension, checked with a golden check | gentle | 4–6 min |
| **ENG1** Measure, don't guess | profile first, then speed up the hotspot only (math.dist), and check the answers stay the same | moderate | 10–15 min |
| **SCI1** Shrink the image | a comprehension with two for clauses: 28 × 28 to 14 × 14, and what it does to speed and accuracy | moderate | 10–15 min |
| **BLD1** Report card | a dict comprehension, and an aligned f-string table ranked with enumerate | moderate | 10–15 min |

In [ ]:
# ▶️ JUST RUN · tools for the OPTIONAL and ADVANCED tasks (run this once)
import math
import time
from labkit import side_by_side
from labkit.profile import profile_call
from labkit.lab2_tools import race_distances, compare_sizes, reader_predictions, reader_with

### OPTIONAL · C6 · Scale the pixels · ✏️ WRITE CODE

Many models (Lab 5's neural network among them) want pixels from 0.0 to 1.0. The intern's normalise uses a while loop; one list comprehension does the same.

**What you'll learn:** a list comprehension, checked with a golden check against the legacy code (Theory S4) · **Difficulty:** gentle · **Typical time:** 4–6 min

**Signature**

```python
def normalise(image: list[int]) -> list[float]
```

**Toy example**

```python
normalise([0, 51, 255])    # [0.0, 0.2, 1.0]: each pixel divided by 255
```

**Square brackets matter:** `[pixel / 255 for pixel in image]` is a list. With round brackets it is a generator: a recipe that hasn't run yet.

**Stuck for 7 minutes after Hint 2?** Run `normalise = rescue("C6")` in a new cell, carry on, and come back to C6 later.

In [ ]:
# ===== YOUR CODE HERE =====
# ✏️ WRITE CODE: replace `pass` with your code, then run this cell (Shift+Enter).
def normalise(image: list[int]) -> list[float]:
    """Return every pixel divided by 255, from 0.0 (paper) to 1.0 (ink), as a new list.

    >>> normalise([0, 51, 255])
    [0.0, 0.2, 1.0]
    """
    # TODO: one list comprehension: pixel / 255 for every pixel of the image
    pass

In [ ]:
# ▶️ JUST RUN · check your work
check("C6", normalise);

In [ ]:
# ▶️ JUST RUN · the golden check: the legacy normalise and yours on the same 20 test images
if ready("normalise"):
    compare(legacy.normalise, normalise)
    row = normalise(test_images[0])[7 * 28 + 6: 7 * 28 + 14]      # 8 pixels of row 7, scaled
    print("Part of row 7 of test image #0, scaled by your normalise:", [round(v, 2) for v in row])

<details>
<summary><strong>Hint 1 · Nudge</strong></summary>

A list comprehension (Theory S4): [expression for pixel in image]. The expression is the new value of one pixel.

</details>

<details>
<summary><strong>Hint 2 · Plan</strong></summary>

1. The new value of one pixel is `pixel / 255` (a single `/`: `//` would round down to 0 or 1).
2. Put it in a list comprehension over `image` and return it.

With a loop (it passes too): `result = []`, then `for pixel in image: result.append(pixel / 255)`, then `return result`.

</details>

<details>
<summary><strong>Hint 3 · Ask an AI reviewer (keep only what you can explain)</strong></summary>

Copy this into your AI assistant: *"I'm learning Python. Act as a code reviewer and do NOT rewrite my code. Here is my function and what the check or my test printed: [paste]. Point out one problem at a time and ask me how I would fix it."*

</details>

### OPTIONAL · ENG1 · Measure, don't guess · Engineer track · ✏️ WRITE CODE

Your reader is readable. Is it slow, and if so, where? Profile one prediction, then speed up the hotspot, and only the hotspot.

**What you'll learn:** profile first, then speed up the hotspot only (math.dist), and check the answers stay the same · **Difficulty:** moderate · **Typical time:** 10–15 min

**Signature**

```python
def fast_distance(a: list[int], b: list[int]) -> float
```

**Toy example**

```python
fast_distance([0, 0], [3, 4])    # math.dist: the square root of 9 + 16 -> 5.0
```

1. **Profile** (the cell below): where does one `predict_knn` spend its time? A *profiler* runs it once and
   times each of your functions (the built-ins they use, such as `sum` and `zip`, count as part of them). The
   top line, the *hotspot*, is where the time goes, so it is the only place where a speed-up pays off.
2. **Write `fast_distance`** with `math.dist(a, b)`. It returns the square root of the squared distance; a square
   root never changes which of two distances is smaller, so the nearest neighbours stay the same.
3. **Measure** with the cells under the check: one distance three ways, then the whole reader with each
   distance. Same answers? How much faster?

**Stuck for 7 minutes after Hint 2?** Run `fast_distance = rescue("ENG1")` in a new cell, carry on, and come back to ENG1 later.

In [ ]:
# ▶️ JUST RUN · step 1: where does ONE prediction spend its time? (about 2 s: the profiler slows code down)
if ready("predict_knn"):
    profile_call(predict_knn, test_images[0], train_images, train_labels)

> **New tool: math.dist**
>
> `math.dist(a, b)` is the straight-line (Euclidean) distance between two points with any number of coordinates:
> the square root of the sum of the squared differences. Its loop runs in C, inside Python itself.
>
> ```python
> import math
> math.dist([0, 0], [3, 4])        # 5.0: the square root of 9 + 16
> ```

In [ ]:
# ===== YOUR CODE HERE =====
# ✏️ WRITE CODE: replace `pass` with your code, then run this cell (Shift+Enter).
import math


def fast_distance(a: list[int], b: list[int]) -> float:
    """Return the Euclidean distance between two images, computed in C by math.dist.

    >>> fast_distance([0, 0], [3, 4])
    5.0
    """
    # TODO: one line with math.dist
    pass

In [ ]:
# ▶️ JUST RUN · check your work
check("ENG1", fast_distance);

In [ ]:
# ▶️ JUST RUN · step 3a: one distance between two real images, three ways (under 1 s)
# fast_distance returns the square root, so its answer differs on purpose: compare the times
if ready("fast_distance"):
    time_versions({"legacy (while loop)": legacy.squared_distance, "yours (sum and zip)": squared_distance,
                   "fast_distance (math.dist)": fast_distance}, test_images[0], train_images[0],
                  compare_answers=False)

In [ ]:
# ▶️ JUST RUN · step 3b: the same k-NN reader with each distance, on 20 test images: same answers?
# (about 2 s; 6 s on Python 3.10)
if ready("squared_distance", "fast_distance"):
    race_distances(squared_distance, fast_distance, test_images[:20], train_images, train_labels)

<details>
<summary><strong>Hint 1 · Nudge</strong></summary>

Read the New tool box: math.dist(a, b) does the whole distance in one call. Remember import math.

</details>

<details>
<summary><strong>Hint 2 · Plan</strong></summary>

1. `import math` (at the top of the cell; it is in the standard library).
2. `return math.dist(a, b)`: the straight-line distance, the square root of what `squared_distance` returns.

</details>

<details>
<summary><strong>Hint 3 · Ask an AI reviewer (keep only what you can explain)</strong></summary>

Copy this into your AI assistant: *"I'm learning Python. Act as a code reviewer and do NOT rewrite my code. Here is my function and what the check or my test printed: [paste]. Point out one problem at a time and ask me how I would fix it."*

</details>

### Explain your result (ENG1)

Edit this cell. A few sentences are enough.

- What was the hotspot, and how many times did it run for one prediction?
- How much faster was the reader with `fast_distance`? Were the predictions identical, and why must they be?
- Would you have guessed the hotspot without the profiler?

### OPTIONAL · SCI1 · Shrink the image · Scientist track · ✏️ WRITE CODE

A 14 × 14 image has 4 times fewer pixels than a 28 × 28 one. Write the canonical shrink, then measure what it does to speed and accuracy.

**What you'll learn:** a comprehension with two for clauses: 28 × 28 to 14 × 14, and what it does to speed and accuracy (Theory S4) · **Difficulty:** moderate · **Typical time:** 10–15 min

**Signature**

```python
def downsample(image: list[int]) -> list[int]
```

**Toy example**

```python
# each new pixel is the floor average of one 2 x 2 block: (a + b + c + d) // 4
# the block whose top-left pixel is at position i holds i, i + 1 (right), i + 28 (below), i + 29
downsample([4] * 784)[:3]    # [4, 4, 4]: a flat image stays flat, and 196 pixels long
```

The kit's 14 × 14 files were made with exactly this rule, so your function must match them pixel for pixel (the check compares). **Readable comprehensions:** at most two `for` clauses and about 80 characters per line; otherwise split them into two steps (the hints show one way).

**Stuck for 7 minutes after Hint 2?** Run `downsample = rescue("SCI1")` in a new cell, carry on, and come back to SCI1 later.

In [ ]:
# ===== YOUR CODE HERE =====
# ✏️ WRITE CODE: replace `pass` with your code, then run this cell (Shift+Enter).
def downsample(image: list[int]) -> list[int]:
    """Shrink a 28 x 28 image to 14 x 14: each new pixel is the floor average of a 2 x 2 block.

    >>> downsample([4] * 784)[:3]
    [4, 4, 4]
    """
    # TODO 1: the top-left position i = r * 28 + c of every block: r and c go 0, 2, 4, ..., 26
    # TODO 2: for each block, (image[i] + image[i + 1] + image[i + 28] + image[i + 29]) // 4
    pass

In [ ]:
# ▶️ JUST RUN · check your work
check("SCI1", downsample);

In [ ]:
# ▶️ JUST RUN · the same test digit at 28 x 28 and at 14 x 14, shrunk by YOUR downsample
if ready("downsample"):
    side_by_side([test_images[0], downsample(test_images[0])], ["28 x 28", "14 x 14 (your downsample)"])

In [ ]:
# ▶️ JUST RUN · the same reader (YOUR predict_knn) on 28 x 28 and on 14 x 14 images: time and answers
# (about 2 s; 7 s on Python 3.10)
if ready("downsample", "predict_knn"):
    compare_sizes(predict_knn, downsample, test_images[:20], test_labels[:20], train_images, train_labels)

<details>
<summary><strong>Hint 1 · Nudge</strong></summary>

Two steps, each one comprehension: first the top-left position of every 2 x 2 block, then one new pixel per block. A row of the big image is 28 pixels long, so the pixel below i is i + 28.

</details>

<details>
<summary><strong>Hint 2 · Plan</strong></summary>

1. `corners = [r * 28 + c for r in range(0, 28, 2) for c in range(0, 28, 2)]`: 14 × 14 = 196 top-left positions, row by row.
2. For each `i` in `corners`: `(image[i] + image[i + 1] + image[i + 28] + image[i + 29]) // 4`, in a second comprehension.

With loops (it passes too): `for r in range(0, 28, 2):` and inside it `for c in range(0, 28, 2):`, appending one value per block.

</details>

<details>
<summary><strong>Hint 3 · Ask an AI reviewer (keep only what you can explain)</strong></summary>

Copy this into your AI assistant: *"I'm learning Python. Act as a code reviewer and do NOT rewrite my code. Here is my function and what the check or my test printed: [paste]. Point out one problem at a time and ask me how I would fix it."*

</details>

### Explain your result (SCI1)

- How many times faster was the 14 × 14 reader? Is that what you expected from 4 times fewer pixels?
- How many answers changed? Did the accuracy on these 20 images go up, down, or stay the same?
- When would you choose the smaller images?

### OPTIONAL · BLD1 · Report card · Builder track · ✏️ WRITE CODE

Ops want one table per run: how well is each digit read, the worst first? Two functions: the numbers, then the table.

**What you'll learn:** a dict comprehension, and an aligned f-string table ranked with enumerate (Theory S4) · **Difficulty:** moderate · **Typical time:** 10–15 min

**Signature**

```python
def per_digit_accuracy(predictions: list[int], truth: list[int]) -> dict[int, float]
def report_card(acc: dict[int, float]) -> str
```

**Toy example**

```python
per_digit_accuracy([7, 1, 1, 3], [7, 7, 1, 3])   # {1: 1.0, 3: 1.0, 7: 0.5}: the two 7s were read as 7 and 1
print(report_card({7: 0.5, 1: 1.0}))
# Rank  Digit  Accuracy
#    1      7     50.0%
#    2      1    100.0%
```

**The table, exactly:** a header line `f"{'Rank':>4}  {'Digit':>5}  {'Accuracy':>8}"`, then one line per digit,
the worst first (equal accuracies: the smaller digit first): `f"{rank:>4}  {digit:>5}  {accuracy:>8.1%}"`, with
the rank from `enumerate(..., start=1)`; the lines joined with `"\n"`. `>4` right-aligns in 4 characters, and
`.1%` shows 0.5 as 50.0%. Each digit is judged on the images whose **true** label it is. Two functions, one
check: `check("BLD1", per_digit_accuracy)` checks `per_digit_accuracy` first, then `report_card`.

**Stuck for 7 minutes after Hint 2?** Run `per_digit_accuracy = rescue("BLD1")` and `from labkit.reference.lab2 import report_card` in a new cell, carry on, and come back to BLD1 later.

In [ ]:
# ===== YOUR CODE HERE =====
# ✏️ WRITE CODE: replace `pass` with your code, then run this cell (Shift+Enter).
def per_digit_accuracy(predictions: list[int], truth: list[int]) -> dict[int, float]:
    """Return {digit: fraction read correctly} for every digit that appears in truth.

    >>> per_digit_accuracy([7, 1, 1, 3], [7, 7, 1, 3])
    {1: 1.0, 3: 1.0, 7: 0.5}
    """
    # TODO: a dict comprehension over the digits of truth: {digit: right / total for digit in ...}
    pass

In [ ]:
# ===== YOUR CODE HERE =====
# ✏️ WRITE CODE: replace `pass` with your code, then run this cell (Shift+Enter).
def report_card(acc: dict[int, float]) -> str:
    """Return an aligned table: a header, then one line per digit, the worst first (rank from 1)."""
    # TODO 1: the digits, sorted by accuracy (and by digit for equal accuracies): sorted(acc, key=...)
    # TODO 2: a header line, then one f-string line per digit with enumerate(..., start=1)
    # TODO 3: return the lines joined with "\n"
    pass

In [ ]:
# ▶️ JUST RUN · check your work
check("BLD1", per_digit_accuracy);   # checks per_digit_accuracy, then report_card

In [ ]:
# ▶️ JUST RUN · the report card of a k-NN reader (k = 3) on 500 test images: a provided fast reader
# (14 x 14 images with math.dist), about 2 s; 3 s on Python 3.10
if ready("per_digit_accuracy", "report_card"):
    predictions, truth = reader_predictions(n_test=500)
    print(report_card(per_digit_accuracy(predictions, truth)))

<details>
<summary><strong>Hint 1 · Nudge</strong></summary>

per_digit_accuracy: for each digit in set(truth), count the images of that digit and how many of them were read right. report_card: sorted() with a key gives the order; enumerate(start=1) gives the ranks.

</details>

<details>
<summary><strong>Hint 2 · Plan</strong></summary>

1. `per_digit_accuracy`: `pairs = list(zip(predictions, truth))`; for each `digit` in `sorted(set(truth))`, right = the pairs with `t == digit and p == t`, total = the pairs with `t == digit`; build `{digit: right / total ...}`.
2. `report_card`: `order = sorted(acc, key=lambda digit: (acc[digit], digit))`; start `lines` with the header; `for rank, digit in enumerate(order, start=1):` append the f-string line of the card; `return "\n".join(lines)`.

</details>

<details>
<summary><strong>Hint 3 · Ask an AI reviewer (keep only what you can explain)</strong></summary>

Copy this into your AI assistant: *"I'm learning Python. Act as a code reviewer and do NOT rewrite my code. Here is my function and what the check or my test printed: [paste]. Point out one problem at a time and ask me how I would fix it."*

</details>

## ADVANCED: for students who move faster

For students who move faster. These tasks are more open: a sanity check plus a cell where you explain your result. Nothing here is needed for the next lab.

| Task | What you'll learn | Difficulty | Typical time |
|---|---|---|---|
| **ENG2** Bugs become tests | regression tests: one assert per bug of the AI's code, and how many buggy readers your test catches | demanding | 15–20 min |
| **SCI2** Choose k with cross-validation | k-fold cross-validation: slicing, a dict of lists, and how much a score moves by chance | demanding | 15–25 min |
| **BLD2** Shift it | nested lists and slicing: shifted copies of the training images, and what more data buys | demanding | 20–25 min |
| **Challenge** The Postcode Machine | read whole postcodes, use the list of valid codes, and decide when the machine should ask a person | open-ended | open-ended |

In [ ]:
# ▶️ JUST RUN · tools for the ADVANCED tasks (run this once)
import math
import time
from labkit.lab2_tools import reader_with
from labkit.ai_samples.lab2_knn_ai import knn_predict_ai as original_ai     # the AI's code, bugs and all (ENG2)
from labkit.mutants.lab2 import AI_BUG_MUTANTS                             # three readers, one AI bug each

### ADVANCED · ENG2 · Bugs become tests · Engineer track · ✏️ WRITE CODE

A bug you fixed once should never come back. Turn each bug of the AI's code into an assert: a regression test for any k-NN reader.

**What you'll learn:** regression tests: one assert per bug, and how many buggy readers they catch · **Difficulty:** demanding · **Typical time:** 15–20 min

**Signature**

```python
def test_knn(predict_fn) -> None
```

**Toy example**

```python
test_knn(predict_knn)       # your reader: every assert passes, nothing is printed
test_knn(original_ai)       # the AI's original knn_predict_ai (from the ADVANCED tools cell): an assert fails
```

Use tiny worlds of 1-pixel images, where you can work out the right answer by hand, like the toy tests of the AI
check. **One assert per AI bug:** for each bug, an input where that bug alone gives a wrong answer. The check runs
your test on a correct reader (it must pass), on the AI's original (it must fail), on three readers with one AI bug
each (it must catch all three) and on three more buggy readers, and says how many it caught. **Why count buggy
readers?** A test that passes on everything proves nothing: the more buggy readers it catches, the more it
protects your code the next time someone changes it. (Testers call this count the *mutation score*.)

In [ ]:
# ===== YOUR CODE HERE =====
# ✏️ WRITE CODE: replace `pass` with your code, then run this cell (Shift+Enter).

def test_knn(predict_fn) -> None:
    """Regression tests for a k-NN reader: one assert per bug of the AI's knn_predict_ai."""
    # TODO: one assert per AI bug (sorting by label, skipping the nearest, max of the labels), in this form:
    #       assert predict_fn([5], [[4], [9]], [1, 2], k=1) == 1, "the nearest image is [4], label 1"
    pass

In [ ]:
# ▶️ JUST RUN · check your work
check("ENG2", test_knn);

In [ ]:
# ▶️ JUST RUN · your test on your reader, on the AI's original, and on the three readers with one AI bug each
if ready("test_knn", "predict_knn"):
    run_test(test_knn, predict_knn, "your predict_knn")
    run_test(test_knn, original_ai, "the AI's original knn_predict_ai", buggy=True)
    for description, buggy in AI_BUG_MUTANTS.items():
        run_test(test_knn, buggy, description, buggy=True)

<details>
<summary><strong>Hint 1 · Nudge</strong></summary>

For each bug, ask: on which tiny input does THIS bug give a wrong answer, while the right reader doesn't? The three toy tests of the AI check are a good start.

</details>

<details>
<summary><strong>Hint 2 · Plan</strong></summary>

1. Sorting by label: k = 1, where the nearest image has a BIGGER label than a farther one.
2. Skipping the nearest: k = 1, with a training image identical to the test image.
3. max(votes): k = 3, where the winning label is smaller than the label of the losing neighbour.
4. To catch the other buggy readers too, also test the tie rules: equal distances (the earlier image wins) and a 1-1 vote (the closer neighbour wins).
5. Check that k is really used: k = 1 on an input where the 3 nearest would vote for a different label.

</details>

<details>
<summary><strong>Hint 3 · Ask an AI reviewer (keep only what you can explain)</strong></summary>

Copy this into your AI assistant: *"I'm learning Python. Act as a code reviewer and do NOT rewrite my code. Here is my function and what the check or my test printed: [paste]. Point out one problem at a time and ask me how I would fix it."*

</details>

### Explain your result (ENG2)

- How many buggy readers did your test catch? Which ones survived, and which input would catch them?
- Why must a test pass on a correct reader before its failures mean anything?

### ADVANCED · SCI2 · Choose k with cross-validation · Scientist track · ✏️ WRITE CODE

k = 3 was the intern's guess. To choose k, you can't use the test set: it must stay unseen until the end. So you hold back part of the training images and score each k on them. But one part is only 100 images, and another part might pick another k. Cross-validation holds back each part in turn: 5 scores per k, so you see how much a score moves by chance.

**What you'll learn:** k-fold cross-validation: slicing, a dict of lists, and how much a score moves by chance · **Difficulty:** demanding · **Typical time:** 15–25 min

**Signature**

```python
def cross_validate(images: list[list[int]], labels: list[int], ks: tuple[int, ...] = (1, 3, 5, 7, 9),
                   folds: int = 5) -> dict[int, list[float]]
```

**Toy example**

```python
# 6 images and folds=3, so size = 6 // 3 = 2 images per fold:
#   fold 0 validates on images[0:2] and learns from images[2:]
#   fold 1 validates on images[2:4] and learns from images[:2] + images[4:]
#   fold 2 validates on images[4:6] and learns from images[:4]
cross_validate(images, labels, ks=(1, 3), folds=3)   # {1: [3 accuracies], 3: [3 accuracies]}
```

**Why not just try every k on the test set?** You would keep the k that happens to suit those 100 images, and
the score would look better than the reader really is on new digits.

**The rule:** with `size = len(images) // folds`, fold `f` validates on the images from `f * size` to
`(f + 1) * size` and learns from all the others (two slices joined with `+`). The test set is never touched.

**A speed trick:** the k nearest neighbours are the first k of the 9 nearest (the list is sorted), so call your
`k_nearest` once per validation image with `max(ks)`, then `majority_vote(neighbours[:k])` for every k: five
times less work.

In [ ]:
# ===== YOUR CODE HERE =====
# ✏️ WRITE CODE: replace `pass` with your code, then run this cell (Shift+Enter).

def cross_validate(images: list[list[int]], labels: list[int], ks: tuple[int, ...] = (1, 3, 5, 7, 9),
                   folds: int = 5) -> dict[int, list[float]]:
    """Return {k: [accuracy on fold 0, accuracy on fold 1, ...]}: k-fold cross-validation (never the test set)."""
    # TODO 1: size = len(images) // folds; scores = {k: [] for k in ks}
    # TODO 2: for each fold: the images it validates on (a slice) and the ones to learn from (two slices + )
    # TODO 3: k_nearest once per validation image with max(ks); for each k, append the fold's accuracy
    pass

In [ ]:
# ▶️ JUST RUN · check your work
check("SCI2", cross_validate);

In [ ]:
# ▶️ JUST RUN · 5-fold cross-validation on 500 training images (14 x 14): 5 scores per k
# (with the speed trick: about 4 s on Python 3.14, 13 s on 3.10; up to a minute without it)
cv_scores = {}
if ready("cross_validate", "k_nearest", "majority_vote"):
    cv_images, cv_labels = load_digits("data/mnist_train_10k_14x14.csv.gz", limit=500)
    cv_scores = cross_validate(cv_images, cv_labels)
    means = {k: round(sum(scores) / len(scores), 4) for k, scores in cv_scores.items()}
    for k, scores in cv_scores.items():
        folds_text = "  ".join(f"{score:.0%}" for score in scores)
        print(f"k = {k}:  folds {folds_text}   mean {means[k]:.1%}   "
              f"spread {min(scores):.0%} to {max(scores):.0%}")
    winners = [max(cv_scores, key=lambda k: cv_scores[k][fold]) for fold in range(5)]
    print("The k that each fold alone would choose:", winners)
    best_k = max(means, key=means.get)          # the best mean (a tie: the smaller k)
    print("Chosen by cross-validation (the best mean): k =", best_k)

In [ ]:
# ▶️ JUST RUN · now, and only now, the test set: once, with the k you chose (100 test images, 14 x 14)
# (about 1 s; 4 s on Python 3.10)
if cv_scores:
    small_test, _ = load_digits("data/mnist_test_2k_14x14.csv.gz", limit=100)
    guesses = [predict_knn(image, cv_images, cv_labels, k=best_k) for image in small_test]
    right = sum(1 for guess, truth in zip(guesses, test_labels) if guess == truth)
    print(f"Test accuracy with k = {best_k}, learning from all 500 images: {right} of {len(test_labels)}")
else:
    print("Run the cross-validation cell above first.")

<details>
<summary><strong>Hint 1 · Nudge</strong></summary>

One loop over the folds. Inside it: slice out the fold, join the rest with +, find the neighbours once (with max(ks)), then append one accuracy per k.

</details>

<details>
<summary><strong>Hint 2 · Plan</strong></summary>

1. `size = len(images) // folds` and `scores = {k: [] for k in ks}`.
2. `for fold in range(folds):` with `start, stop = fold * size, (fold + 1) * size`: learn from `images[:start] + images[stop:]` (and the same slices of `labels`), validate on `images[start:stop]`.
3. `nearest = [k_nearest(image, fit_images, fit_labels, max(ks)) for image in images[start:stop]]`; for each k, count the `(neighbours, truth)` pairs of `zip(nearest, labels[start:stop])` where `majority_vote(neighbours[:k]) == truth`, divide by `size`, and append it to `scores[k]`.

</details>

<details>
<summary><strong>Hint 3 · Ask an AI reviewer (keep only what you can explain)</strong></summary>

Copy this into your AI assistant: *"I'm learning Python. Act as a code reviewer and do NOT rewrite my code. Here is my function and what the check or my test printed: [paste]. Point out one problem at a time and ask me how I would fix it."*

</details>

### Explain your result (SCI2)

- Which k had the best mean? Is its lead bigger than the spread between the folds?
- Did every fold, on its own, choose the same k? What does that say about choosing k on one validation split?
- Why is the test set still used only once, at the very end?

### ADVANCED · BLD2 · Shift it · Builder track · ✏️ WRITE CODE

A digit written one pixel to the left is still the same digit, but not to a pixel-by-pixel reader. Add shifted copies of every training image, and measure what the extra data buys.

**What you'll learn:** nested lists and slicing: shifted copies of the training images, and what more data buys · **Difficulty:** demanding · **Typical time:** 20–25 min

**Signature**

```python
def to_grid(image: list[int], width: int = 28) -> list[list[int]]
def flatten(grid: list[list[int]]) -> list[int]
def shift(image: list[int], dx: int, dy: int, width: int = 28) -> list[int]
```

**Toy example**

```python
to_grid([1, 2, 3, 4, 5, 6], width=3)                 # [[1, 2, 3], [4, 5, 6]]
flatten([[1, 2], [3, 4]])                            # [1, 2, 3, 4]
shift([1, 2, 3, 4, 5, 6, 7, 8, 9], 1, 0, width=3)    # [0, 1, 2, 0, 4, 5, 0, 7, 8]: one to the right
```

`dx > 0` moves the ink right and `dy > 0` moves it down (negative: left and up). What moves out of the image is
lost; the new pixels are paper (0). Work on the grid: a row slice such as `row[:width - dx]`, and zeros added with
`[0] * dx + ...`. Three functions, one check: `check("BLD2", shift)` also checks `to_grid` and `flatten`. Then
`augment` builds the bigger training set, and the experiment measures it.

In [ ]:
# ===== YOUR CODE HERE =====
# ✏️ WRITE CODE: replace `pass` with your code, then run this cell (Shift+Enter).

def to_grid(image: list[int], width: int = 28) -> list[list[int]]:
    """Cut a flat image into rows of `width` pixels."""
    # TODO: one comprehension of row slices
    pass


def flatten(grid: list[list[int]]) -> list[int]:
    """Join the rows back into one flat list."""
    # TODO: a comprehension with two for clauses (rows, then pixels)
    pass


def shift(image: list[int], dx: int, dy: int, width: int = 28) -> list[int]:
    """Move the picture dx pixels right and dy pixels down (negative: left / up); fill with paper (0)."""
    # TODO: to_grid; add blank rows and drop rows for dy; add zeros and cut each row for dx; flatten
    pass

In [ ]:
# ▶️ JUST RUN · check your work
check("BLD2", shift);   # checks shift, then to_grid and flatten

In [ ]:
# ===== YOUR CODE HERE =====
# ✏️ WRITE CODE: replace `pass` with your code, then run this cell (Shift+Enter).

def augment(images: list[list[int]], labels: list[int], width: int = 28) -> tuple[list[list[int]], list[int]]:
    """The images, plus each one shifted by one pixel right, left, down and up: 5 times as many."""
    # TODO: start with copies of images and labels; for each (dx, dy) in (1, 0), (-1, 0), (0, 1), (0, -1),
    #       add every image shifted by (dx, dy), and the labels again
    pass

In [ ]:
# ▶️ JUST RUN · more data against more time: 1,000 training images, then 5,000 with the shifted copies
# (100 test images; the provided reader_with uses math.dist; about 6 s in all; 10 s on Python 3.10)
if ready("augment", "shift"):
    more_images, more_labels = augment(train_images, train_labels)
    for name, images, labels in (("1,000 training images", train_images, train_labels),
                                 (f"{len(more_images):,} with shifted copies", more_images, more_labels)):
        start = time.perf_counter()
        guesses = [reader_with(math.dist, image, images, labels) for image in test_images]
        seconds = time.perf_counter() - start
        right = sum(1 for guess, truth in zip(guesses, test_labels) if guess == truth)
        print(f"{name:<30} {right} of {len(test_labels)} right · {1000 * seconds / len(test_labels):.0f} ms per digit")

<details>
<summary><strong>Hint 1 · Nudge</strong></summary>

to_grid and flatten are one comprehension each. For shift, think in rows: moving down adds blank rows on top and drops rows at the bottom; moving right adds zeros on the left of every row and cuts its end.

</details>

<details>
<summary><strong>Hint 2 · Plan</strong></summary>

1. `to_grid`: `[image[start:start + width] for start in range(0, len(image), width)]`.
2. `flatten`: `[pixel for row in grid for pixel in row]`.
3. `shift`: `grid = to_grid(image, width)`; for `dy >= 0`, `rows = [[0] * width for _ in range(dy)] + grid[:len(grid) - dy]` (for `dy < 0`, `grid[-dy:]` plus blank rows at the end); then for `dx >= 0`, `[[0] * dx + row[:width - dx] for row in rows]` (for `dx < 0`, `row[-dx:] + [0] * -dx`); return `flatten(rows)`.

</details>

<details>
<summary><strong>Hint 3 · Ask an AI reviewer (keep only what you can explain)</strong></summary>

Copy this into your AI assistant: *"I'm learning Python. Act as a code reviewer and do NOT rewrite my code. Here is my function and what the check or my test printed: [paste]. Point out one problem at a time and ask me how I would fix it."*

</details>

### Explain your result (BLD2)

- How many more test digits were read correctly with the shifted copies? How much slower was each digit?
- Is that a good trade for Garabato? What would you try instead of 5 times more data?

### ADVANCED · Challenge: The Postcode Machine · ✏️ WRITE CODE

**What you'll learn:** reading a whole postcode, using what you know about valid answers, and turning a threshold into a business decision · **Difficulty:** open-ended · **Typical time:** open-ended (you can continue at home)

Ops want postcodes, not digits. A postcode strip is 5 handwritten digits; `data/postcodes_test.csv` has 300 strips (made from the kit's test images) and `data/valid_postcodes.txt` the postcodes Garabato delivers to (a simplified list: 28001 to 28055). **Stage 1:** read each strip digit by digit, then constrained to the valid postcodes. **Stage 2 (open):** let the machine say *I'm not sure* and send the parcel to a person, and choose when, in euros. AI is your pair programmer here: verify what it suggests, and say how you used it in your Engineering log.

**Ideas to try**

- The provided `digit_scores(image, centroids)` gives, for one digit image, its distance to the average image of each digit 0 to 9: 10 numbers, the lowest is the most alike.
- Constrained decoding: of all the valid codes, pick the one whose 5 digits have the smallest total score. That is one `min()` over the valid codes, with a key that adds up `scores[i][int(ch)]` for every position `i` and character `ch` of the code (`enumerate` gives both). Use `sorted(valid)`: a set has no fixed order, so a tie would otherwise go to a different code from run to run.
- Stage 2: compare the best total with the second best. If they are closer than a margin, the machine isn't sure: return None.
- Cost model: a manual review costs €0.35 per parcel, a misrouted parcel €7.80. Which margin gives the lowest cost per 1,000 parcels? Find it with `min(costs, key=costs.get)`, try margins of your own (250, 350, or `float("inf")`, which sends every parcel to a person), and compare.

**Rules**

- Standard library only. Train on the kit's training images; the postcode strips are for measuring.
- The postcode list is simplified, and Garabato Parcels is fictional.

In [ ]:
# ▶️ JUST RUN · the postcode strips, the valid postcodes and the digit scores (about 2 s)
from labkit import load_postcodes, load_valid_postcodes
from labkit.reference.lab1 import fit_centroids, predict_centroid      # Lab 1's average digits (SCI1)
from labkit.lab2_tools import digit_scores, strip_digits, show_strip, cost_table
all_train_images, all_train_labels = load_digits("data/mnist_train_10k.csv.gz")
all_test_images, all_test_labels = load_digits("data/mnist_test_2k.csv.gz")
centroids = fit_centroids(all_train_images, all_train_labels)      # one average image per digit
strips = load_postcodes()            # 300 strips: {"strip_id", "postcode", "indices"}
valid = load_valid_postcodes()       # the valid postcodes, as a set of strings
scored_strips = [([digit_scores(image, centroids) for image in strip_digits(strip, all_test_images)],
                  strip["postcode"]) for strip in strips]
print(f"{len(strips)} postcode strips · {len(valid)} valid postcodes, from {min(valid)} to {max(valid)}")
show_strip(strip_digits(strips[0], all_test_images), f"Strip {strips[0]['strip_id']}: the true postcode is {strips[0]['postcode']}")
print("Its first digit's scores (digits 0 to 9; lower = more alike):", [round(s) for s in scored_strips[0][0][0]])

In [ ]:
# ===== YOUR CODE HERE =====
# ✏️ WRITE CODE: replace `pass` with your code, then run this cell (Shift+Enter).

def read_postcode(digit_images: list[list[int]], predict) -> str:
    """Read a postcode digit by digit: predict(image) gives each digit; join them into a string."""
    # TODO: one line with "".join(...) and str(...)
    pass

In [ ]:
# ▶️ JUST RUN · stage 1a: every strip read digit by digit, each digit the closest average digit (about 1 s)
if ready("read_postcode"):
    said = [read_postcode(strip_digits(strip, all_test_images), lambda image: predict_centroid(image, centroids))
            for strip in strips]
    exact = sum(1 for postcode, strip in zip(said, strips) if postcode == strip["postcode"])
    print(f"Digit by digit: {exact} of {len(strips)} postcodes exactly right; "
          f"{sum(1 for postcode in said if postcode in valid)} of the answers are valid postcodes at all.")

In [ ]:
# ===== YOUR CODE HERE =====
# ✏️ WRITE CODE: replace `pass` with your code, then run this cell (Shift+Enter).

def read_postcode_constrained(scores: list[list[float]], valid: set[str]) -> str:
    """The valid postcode with the smallest total score (scores[i][d]: how unlike digit d image i is)."""
    # TODO: one min() over sorted(valid), with a key that adds up scores[i][int(ch)] for every position i
    pass

In [ ]:
# ▶️ JUST RUN · check your work
check("CH", read_postcode);   # checks read_postcode, then read_postcode_constrained

In [ ]:
# ▶️ JUST RUN · stage 1b: every strip read as the most likely VALID postcode
if ready("read_postcode_constrained"):
    exact = sum(1 for scores, truth in scored_strips if read_postcode_constrained(scores, valid) == truth)
    print(f"Constrained to valid postcodes: {exact} of {len(strips)} exactly right.")

#### Stage 2 (open): when should the machine ask a person?

A wrong postcode sends the parcel to the wrong depot. Return `None` when the machine isn't sure: the best two
valid codes have totals closer than `margin`. Then the table below counts, for several margins, the parcels
read right, misrouted, and sent to manual review, and what that costs per 1,000 parcels.

In [ ]:
# ===== YOUR CODE HERE =====
# ✏️ WRITE CODE: replace `pass` with your code, then run this cell (Shift+Enter).

def read_postcode_or_reject(scores: list[list[float]], valid: set[str], margin: float) -> str | None:
    """Like read_postcode_constrained, but None when the best two valid codes are closer than `margin`."""
    # TODO: every valid code with its total, sorted; compare the best two totals with the margin
    pass

In [ ]:
# ▶️ JUST RUN · the cost of each margin: €0.35 per manual review, €7.80 per misrouted parcel
costs = {}                          # {margin: cost per 1,000 parcels}
if ready("read_postcode_or_reject"):
    costs = cost_table(lambda scores, margin: read_postcode_or_reject(scores, valid, margin), scored_strips,
                       margins=[0, 50, 100, 150, 200, 300, 400])

In [ ]:
# ===== YOUR CODE HERE =====
# ✏️ WRITE CODE: replace `pass` with your code, then run this cell (Shift+Enter).
# TODO 1: the cheapest margin of the table, and its cost: min(costs, key=...)
# TODO 2: margins of your own with cost_table(...), e.g. [250, 350, float("inf")] (inf: review every parcel)
pass

**Your recommendation to Ops** (one or two sentences, edit this cell): which margin would you choose, what
does it cost per 1,000 parcels, and what would change your mind? *A threshold is a business decision:* the
same reader gives a different best margin if a misroute costs €2 instead of €7.80.

### Engineering log

Short notes are fine. Edit this cell as you go.

- **What I tried:**
- **What I measured:**
- **What I'd do next:**

## AI-use declaration (about 1 minute)

This course asks you to use AI critically and to say how you used it. This cell stays in your notebook:
nobody collects or grades it. Please don't write your name.

- **Tools I used** (or "none"):
- **For which tasks** (e.g. C2, ENG1):
- **One prompt I used** (copy it, or summarise it in one line):
- **How I checked the answer** (check(), my own test, reading the code, a classmate…):
- **Something the AI got wrong, or that I changed:**
- **One line of AI-written code I can now explain in my own words:**

## Before you go (private, 30 seconds)
- One thing I can do now that I couldn't at the start of this lab:
- One thing I'm still unsure about:

## What's next

- **Catch-up pack**, on IE Campus after class: `lab2_solution.ipynb` (CORE and OPTIONAL solutions), `lab2_express.ipynb` (the minimum path in about 25 minutes, with the same checks) and `lab2_what_you_missed` (a one-page summary).
- **ADVANCED solutions** (including the Challenge) come with the Lab 3 kit, in `solutions_previous/`, so the Challenge stays open until then.
- **Next:** Lab 3 · The Scanner Mess, Wed 11 Nov, 15:00–16:20, MMB-303. The kit appears on IE Campus 48 hours before.
- **The League** is opt-in and uses pseudonyms: the rules are in `LEAGUE.md`.
- **Missed something?** `labkit/reference/lab2.py` has a clear, commented version of every CORE and OPTIONAL function of this lab; `labkit/reference/lab1.py` has Lab 1's.
- **Lab 1's ADVANCED solutions** are in your kit, in `solutions_previous/` (the full solution notebook and `lab1_advanced_solutions.py`).
- **For the three weeks to Lab 3**, also on IE Campus: `bridge_2to3.ipynb` (about 25 minutes, CSV, JSON and strings, with checks) and `holiday_quest.ipynb`. Copy both into your `lab2_kit` folder, next to `labkit`, and run their first cell.